# Laboratorio #3

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab4)

## Librerías y constantes

In [56]:
# !pip install openeo geopandas

In [ ]:
import openeo
import geopandas as gpd

## Inciso 1

Establecer conexión con Sentibler Hub (Copernicus)

In [58]:
def connectToSentinelHub():
    return openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

## Inciso 2

Leer bounding box desde un GeoJSON

In [59]:
def getBoundingBoxFromGeojson(path):
    gdf = gpd.read_file(path)
    bounds = gdf.total_bounds  # xmin, ymin, xmax, ymax
    return {
        "west": bounds[0],
        "south": bounds[1],
        "east": bounds[2],
        "north": bounds[3]
    }

## Inciso 3

Cargar cubo de Sentinel-2 para el período

In [60]:
def loadSentinelCube(connection, bbox, startDate, endDate, bands):
    return connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[startDate, endDate],
        bands=bands
    )

Descargar cubo como TIFF

In [ ]:
def downloadCubeAsTiff(connection, cube, outputPath):
    result = cube.save_result(format="GTIFF")
    job = connection.create_job(result)
    job.start_and_wait()
    job.download_results(outputPath)


## Inciso 4

Aplicar script de cianobacteria (Evalscript)

In [ ]:
def applyCyanobacteriaEvalscript(connection, bbox, startDate, endDate):
    evalscript = '''
                // CyanoLakes Chlorophyll-a L1C
                // Jeremy Kravitz & Mark Matthews (2020)
                var MNDWI_threshold=0.42;
                var NDWI_threshold=0.4;
                var filter_UABS=true;
                var filter_SSI=false;

                function wbi(r,g,b,nir,swir1,swir2) {
                    let ws=0;
                    try {
                        var ndvi=(nir-r)/(nir+r);
                        var mndwi=(g-swir1)/(g+swir1);
                        var ndwi=(g-nir)/(g+nir);
                        var ndwi_leaves=(nir-swir1)/(nir+swir1);
                        var aweish=b+2.5*g-1.5*(nir+swir1)-0.25*swir2;
                        var aweinsh=4*(g-swir1)-(0.25*nir+2.75*swir1);
                        var dbsi=((swir1-g)/(swir1+g))-ndvi;
                        if (mndwi > MNDWI_threshold || ndwi > NDWI_threshold || aweinsh > 0.1879 || aweish > 0.1112 || ndvi < -0.2 || ndwi_leaves > 1) {
                            ws = 1;
                        }
                        if (filter_UABS && ws==1) {
                            if ((aweinsh<=-0.03)||(dbsi>0)) {ws=0;}
                        }
                    } catch(err) {ws=0;}
                    return ws;
                }

                function setup() {
                    return {
                        input: ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"],
                        output: { bands: 3 }
                    };
                }

                function evaluatePixel(sample) {
                    let water = wbi(sample.B04,sample.B03,sample.B02,sample.B08,sample.B11,sample.B12);
                    function FAI (a,b,c) {return (b-a-(c-a)*(783-665)/(865-665));}
                    let FAIv = FAI(sample.B04,sample.B07,sample.B8A);
                    function NDCI (a,b) {return (b-a)/(b+a);}
                    let NDCIv = NDCI(sample.B04,sample.B05);
                    let chl = 826.57 * Math.pow(NDCIv, 3) - 176.43 * Math.pow(NDCIv, 2) + 19 * NDCIv + 4.071;

                    if (water==0) return [3*sample.B04,3*sample.B03,3*sample.B02];
                    else if (FAIv>0.08) return [233/255,72/255,21/255];
                    else if (chl<0.5) return [0,0,1.0];
                    else if (chl<1) return [0,0,1.0];
                    else if (chl<2.5) return [0,59/255,1];
                    else if (chl<3.5) return [0,98/255,1];
                    else if (chl<5) return [15/255,113/255,141/255];
                    else if (chl<7) return [14/255,141/255,120/255];
                    else if (chl<8) return [13/255,141/255,103/255];
                    else if (chl<10) return [30/255,226/255,28/255];
                    else return [233/255,72/255,21/255];
                }
                '''

    cube = connection.load_collection(
        "SENTINEL2_L1C",
        spatial_extent=bbox,
        temporal_extent=[startDate, endDate]
    ).evalscript(evalscript)

    return cube


Crear y descargar imagen procesada con script

In [ ]:
# Conexión
connection = connectToSentinelHub()

# GeoJSON
atitlanBBox = getBoundingBoxFromGeojson("data/Lago_Atitlan.geojson")
amatitlanBBox = getBoundingBoxFromGeojson("data/Lago_Amatitlan.geojson")

# Fechas
startDate = "2025-04-01"
endDate = "2025-08-01"

# Bandas requeridas
bands = ["B02", "B03", "B04", "B08"]

# Cubos base
atitlanCube = loadSentinelCube(connection, atitlanBBox, startDate, endDate, bands)
amatitlanCube = loadSentinelCube(connection, amatitlanBBox, startDate, endDate, bands)

# Descargas base
downloadCubeAsTiff(connection, atitlanCube, "data/Bandas_Atitlan")
downloadCubeAsTiff(connection, amatitlanCube, "data/Bandas_Amatitlan")

# Cubos con detección de cianobacteria
cyanoAtitlanCube = applyCyanobacteriaEvalscript(connection, atitlanBBox, startDate, endDate)
cyanoAmatitlanCube = applyCyanobacteriaEvalscript(connection, amatitlanBBox, startDate, endDate)

# Descargas procesadas
downloadCubeAsTiff(connection, cyanoAtitlanCube, "data/Cyano_Atitlan")
downloadCubeAsTiff(connection, cyanoAmatitlanCube, "data/Cyano_Amatitlan")

Authenticated using refresh token.
0:00:00 Job 'j-25080722415648e99b75da28ae202249': send 'start'
0:00:13 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:00:18 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:00:25 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:00:33 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:00:43 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:00:55 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:01:11 Job 'j-25080722415648e99b75da28ae202249': queued (progress 0%)
0:01:30 Job 'j-25080722415648e99b75da28ae202249': running (progress N/A)
0:01:54 Job 'j-25080722415648e99b75da28ae202249': running (progress N/A)
0:02:25 Job 'j-25080722415648e99b75da28ae202249': running (progress N/A)
0:03:02 Job 'j-25080722415648e99b75da28ae202249': running (progress N/A)
0:03:49 Job 'j-25080722415648e99b75da28ae202249': running (progress N/A)
0:04:47 Job 'j-25080722415648e99b75da28a

C:\Users\josue\AppData\Local\Temp\ipykernel_45240\446738250.py:5: UserDeprecationWarning: Call to deprecated method download_results. (Instead use `BatchJob.get_results` and the more flexible download functionality of `JobResults`) -- Deprecated since version 0.4.10.
  job.download_results(outputPath)
d:\repositorios\UVG\2025\labs-ds\lab4\.venv\Lib\site-packages\openeo\rest\job.py:199: UserDeprecationWarning: Call to deprecated method get_result. (Use `BatchJob.get_results` instead.) -- Deprecated since version 0.4.10.
  return self.get_result().download_files(target)
d:\repositorios\UVG\2025\labs-ds\lab4\.venv\Lib\site-packages\openeo\rest\job.py:203: UserDeprecationWarning: Call to deprecated class _Result. (Use `JobResults` instead) -- Deprecated since version 0.4.10.
  return _Result(self)


## Inciso 5



### Liberías

In [ ]:
import os
import rasterio
import numpy as np

In [ ]:
def loadTifFilesAsArrays(folderPath):
    tifArrays = []
    dates = []

    for filename in sorted(os.listdir(folderPath)):
        if filename.endswith(".tif"):
            path = os.path.join(folderPath, filename)
            with rasterio.open(path) as src:
                array = src.read()  # (bands, height, width)
                tifArrays.append(array)
                # Extraer fecha del nombre del archivo
                date_str = filename.replace("openEO_", "").replace("Z.tif", "")
                dates.append(date_str)

    return dates, tifArrays


In [ ]:
atitlanDates, atitlanArrays = loadTifFilesAsArrays("data/Bandas_Atitlan")
print("Fechas cargadas:", atitlanDates[:5])
print("Forma de primera imagen:", atitlanArrays[0].shape)

## Inciso 6

## Inciso 7

## Inciso 8

## Inciso 9